# Medical Annotation Database Population

This notebook allows you to manually populate the medical annotation database with documents, entities, and relationships.

## Quick Guide:
1. **Create Documents** - Add medical text documents
2. **Create Annotations** - Set up annotation sessions
3. **Add Entities** - Tag medical entities (diseases, medications, etc.)
4. **Add Relations** - Create relationships between entities
5. **Query & Explore** - View and explore the database


## Setup


In [3]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd()
sys.path.insert(0, str(project_root))

print(f"Project root: {project_root}")
print("Python path updated")


Project root: C:\Users\Jan\projects\oracle-challenge
Python path updated


In [4]:

import pandas as pd
from sqlmodel import Session, select

# Import database
from src.database import create_db_and_tables, get_engine

# Import models
from src.models import (
    Annotation,
    Document,
    Entity,
    EntityType,
    Relation,
    RelationType,
)

print("✅ Imports successful")

engine = get_engine()
print("✅ Engine created")
# Create tables if they don't exist
create_db_and_tables()
print("✅ Database tables created/verified")


✅ Imports successful
✅ Engine created
2025-11-06 07:21:36,687 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-11-06 07:21:36,688 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("annotations")
2025-11-06 07:21:36,689 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-11-06 07:21:36,690 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("documents")
2025-11-06 07:21:36,690 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-11-06 07:21:36,691 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("entities")
2025-11-06 07:21:36,692 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-11-06 07:21:36,693 INFO sqlalchemy.engine.Engine PRAGMA main.table_info("relations")
2025-11-06 07:21:36,694 INFO sqlalchemy.engine.Engine [raw sql] ()
2025-11-06 07:21:36,695 INFO sqlalchemy.engine.Engine COMMIT
✅ Database tables created/verified


## Helper Functions


In [5]:
def create_document(title: str, text: str) -> Document:
    """Create a new document in the database."""
    with Session(engine) as session:
        doc = Document(title=title, text=text)
        session.add(doc)
        session.commit()
        session.refresh(doc)
        print(f"✅ Document created: ID={doc.id}, Title='{doc.title}'")
        return doc


def create_annotation(document_id: int, annotator_id: str = "manual", status: str = "in_progress") -> Annotation:
    """Create a new annotation for a document."""
    with Session(engine) as session:
        annotation = Annotation(
            document_id=document_id,
            annotator_id=annotator_id,
            status=status,
        )
        session.add(annotation)
        session.commit()
        session.refresh(annotation)
        print(f"✅ Annotation created: ID={annotation.id}, Document={document_id}, Status={status}")
        return annotation


def create_entity(
    annotation_id: int,
    text: str,
    entity_type: EntityType,
    start_char: int,
    end_char: int,
    confidence: float = 1.0,
) -> Entity:
    """Create a new entity annotation."""
    with Session(engine) as session:
        entity = Entity(
            annotation_id=annotation_id,
            text=text,
            entity_type=entity_type,
            start_char=start_char,
            end_char=end_char,
            confidence=confidence,
        )
        session.add(entity)
        session.commit()
        session.refresh(entity)
        print(f"✅ Entity created: ID={entity.id}, Text='{entity.text}', Type={entity_type.value}")
        return entity


def create_relation(
    annotation_id: int,
    source_entity_id: int,
    target_entity_id: int,
    relation_type: RelationType,
    confidence: float = 1.0,
) -> Relation:
    """Create a new relation between entities."""
    with Session(engine) as session:
        relation = Relation(
            annotation_id=annotation_id,
            source_entity_id=source_entity_id,
            target_entity_id=target_entity_id,
            relation_type=relation_type,
            confidence=confidence,
        )
        session.add(relation)
        session.commit()
        session.refresh(relation)
        print(f"✅ Relation created: ID={relation.id}, {source_entity_id} --[{relation_type.value}]--> {target_entity_id}")
        return relation


print("✅ Helper functions defined")


✅ Helper functions defined


## Query Functions


In [6]:
def list_documents() -> pd.DataFrame:
    """List all documents in the database."""
    with Session(engine) as session:
        documents = session.exec(select(Document)).all()
        data = [
            {
                "ID": doc.id,
                "Title": doc.title,
                "Text Preview": doc.text[:50] + "..." if len(doc.text) > 50 else doc.text,
                "Text Length": len(doc.text),
                "Created": doc.created_at.isoformat(),
            }
            for doc in documents
        ]
        return pd.DataFrame(data)


def list_annotations(document_id: int | None = None) -> pd.DataFrame:
    """List annotations, optionally filtered by document."""
    with Session(engine) as session:
        statement = select(Annotation)
        if document_id is not None:
            statement = statement.where(Annotation.document_id == document_id)

        annotations = session.exec(statement).all()
        data = [
            {
                "ID": ann.id,
                "Document ID": ann.document_id,
                "Annotator": ann.annotator_id,
                "Status": ann.status,
                "Entities": len(ann.entities),
                "Relations": len(ann.relations),
                "Updated": ann.updated_at.isoformat(),
            }
            for ann in annotations
        ]
        return pd.DataFrame(data)


def list_entities(annotation_id: int | None = None) -> pd.DataFrame:
    """List entities, optionally filtered by annotation."""
    with Session(engine) as session:
        statement = select(Entity)
        if annotation_id is not None:
            statement = statement.where(Entity.annotation_id == annotation_id)

        entities = session.exec(statement).all()
        data = [
            {
                "ID": ent.id,
                "Text": ent.text,
                "Type": ent.entity_type.value,
                "Start": ent.start_char,
                "End": ent.end_char,
                "Confidence": ent.confidence,
                "Annotation ID": ent.annotation_id,
            }
            for ent in entities
        ]
        return pd.DataFrame(data)


def list_relations(annotation_id: int | None = None) -> pd.DataFrame:
    """List relations, optionally filtered by annotation."""
    with Session(engine) as session:
        statement = select(Relation)
        if annotation_id is not None:
            statement = statement.where(Relation.annotation_id == annotation_id)

        relations = session.exec(statement).all()
        data = [
            {
                "ID": rel.id,
                "Source Entity ID": rel.source_entity_id,
                "Relation Type": rel.relation_type.value,
                "Target Entity ID": rel.target_entity_id,
                "Confidence": rel.confidence,
                "Annotation ID": rel.annotation_id,
            }
            for rel in relations
        ]
        return pd.DataFrame(data)


print("✅ Query functions defined")


✅ Query functions defined


## View Database Summary


In [7]:
def database_summary():
    """Print a summary of the database contents."""
    with Session(engine) as session:
        doc_count = len(session.exec(select(Document)).all())
        ann_count = len(session.exec(select(Annotation)).all())
        ent_count = len(session.exec(select(Entity)).all())
        rel_count = len(session.exec(select(Relation)).all())

        print("\n" + "="*50)
        print("📊 DATABASE SUMMARY")
        print("="*50)
        print(f"📄 Documents:   {doc_count}")
        print(f"📝 Annotations: {ann_count}")
        print(f"🏷️  Entities:    {ent_count}")
        print(f"🔗 Relations:   {rel_count}")
        print("="*50 + "\n")

database_summary()


2025-11-06 07:21:36,722 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-11-06 07:21:36,731 INFO sqlalchemy.engine.Engine SELECT documents.id, documents.title, documents.text, documents.created_at, documents.updated_at 
FROM documents
2025-11-06 07:21:36,732 INFO sqlalchemy.engine.Engine [generated in 0.00071s] ()
2025-11-06 07:21:36,734 INFO sqlalchemy.engine.Engine SELECT annotations.id, annotations.annotator_id, annotations.status, annotations.document_id, annotations.created_at, annotations.updated_at 
FROM annotations
2025-11-06 07:21:36,735 INFO sqlalchemy.engine.Engine [generated in 0.00066s] ()
2025-11-06 07:21:36,737 INFO sqlalchemy.engine.Engine SELECT entities.id, entities.text, entities.entity_type, entities.start_char, entities.end_char, entities.confidence, entities.annotation_id, entities.created_at, entities.updated_at 
FROM entities
2025-11-06 07:21:36,738 INFO sqlalchemy.engine.Engine [generated in 0.00109s] ()
2025-11-06 07:21:36,740 INFO sqlalchemy.engine.Engine 

## Example 1: Create Sample Document with Annotations

Let's create a medical case document and add annotations to it.


In [8]:
# Create a sample document
sample_text = """Patient presents with persistent dry cough for 10 days, accompanied by fever and shortness of breath.
Chest X-ray reveals bilateral infiltrates suggestive of pneumonia.
Prescribed azithromycin 500mg daily for 5 days and albuterol inhaler for bronchodilation.
Advised bed rest and increased fluid intake."""

doc = create_document(
    title="Case 1: Respiratory Infection",
    text=sample_text
)

print(f"\nDocument text length: {len(sample_text)} characters")
print(f"Document ID: {doc.id}")


2025-11-06 07:21:36,747 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-11-06 07:21:36,749 INFO sqlalchemy.engine.Engine INSERT INTO documents (title, text, created_at, updated_at) VALUES (?, ?, ?, ?)
2025-11-06 07:21:36,750 INFO sqlalchemy.engine.Engine [generated in 0.00097s] ('Case 1: Respiratory Infection', 'Patient presents with persistent dry cough for 10 days, accompanied by fever and shortness of breath. \nChest X-ray reveals bilateral infiltrates sug ... (11 characters truncated) ... pneumonia. \nPrescribed azithromycin 500mg daily for 5 days and albuterol inhaler for bronchodilation. \nAdvised bed rest and increased fluid intake.', '2025-11-05 21:21:36.746874', '2025-11-05 21:21:36.746894')
2025-11-06 07:21:36,752 INFO sqlalchemy.engine.Engine COMMIT
2025-11-06 07:21:36,755 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-11-06 07:21:36,758 INFO sqlalchemy.engine.Engine SELECT documents.id, documents.title, documents.text, documents.created_at, documents.updated_at 
FR

In [9]:
# Create annotation session for this document
annotation = create_annotation(
    document_id=doc.id,
    annotator_id="doctor_smith",
    status="in_progress"
)

print(f"\nAnnotation ID: {annotation.id}")


2025-11-06 07:21:36,765 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-11-06 07:21:36,767 INFO sqlalchemy.engine.Engine INSERT INTO annotations (annotator_id, status, document_id, created_at, updated_at) VALUES (?, ?, ?, ?, ?)
2025-11-06 07:21:36,767 INFO sqlalchemy.engine.Engine [generated in 0.00093s] ('doctor_smith', 'IN_PROGRESS', 1, '2025-11-05 21:21:36.765187', '2025-11-05 21:21:36.765203')
2025-11-06 07:21:36,770 INFO sqlalchemy.engine.Engine COMMIT
2025-11-06 07:21:36,773 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-11-06 07:21:36,774 INFO sqlalchemy.engine.Engine SELECT annotations.id, annotations.annotator_id, annotations.status, annotations.document_id, annotations.created_at, annotations.updated_at 
FROM annotations 
WHERE annotations.id = ?
2025-11-06 07:21:36,775 INFO sqlalchemy.engine.Engine [generated in 0.00075s] (1,)
✅ Annotation created: ID=1, Document=1, Status=in_progress
2025-11-06 07:21:36,776 INFO sqlalchemy.engine.Engine ROLLBACK

Annotation ID: 1


In [10]:
# Now let's add entities to this annotation
# First, let's identify the positions in the text

# Let's find positions of key terms
text = sample_text

# Find positions using Python string methods
print("Finding entity positions in text...\n")

# Symptom: dry cough
pos = text.find("dry cough")
print(f"'dry cough' at position {pos}-{pos + len('dry cough')}")

# Symptom: fever
pos = text.find("fever")
print(f"'fever' at position {pos}-{pos + len('fever')}")

# Symptom: shortness of breath
pos = text.find("shortness of breath")
print(f"'shortness of breath' at position {pos}-{pos + len('shortness of breath')}")

# Disease: pneumonia
pos = text.find("pneumonia")
print(f"'pneumonia' at position {pos}-{pos + len('pneumonia')}")

# Medication: azithromycin
pos = text.find("azithromycin")
print(f"'azithromycin' at position {pos}-{pos + len('azithromycin')}")

# Dosage: 500mg
pos = text.find("500mg")
print(f"'500mg' at position {pos}-{pos + len('500mg')}")

# Medication: albuterol
pos = text.find("albuterol")
print(f"'albuterol' at position {pos}-{pos + len('albuterol')}")


Finding entity positions in text...

'dry cough' at position 33-42
'fever' at position 71-76
'shortness of breath' at position 81-100
'pneumonia' at position 159-168
'azithromycin' at position 182-194
'500mg' at position 195-200
'albuterol' at position 222-231


In [11]:
# Create entities
entities = []

# Symptoms
entity1 = create_entity(
    annotation_id=annotation.id,
    text="dry cough",
    entity_type=EntityType.SYMPTOM,
    start_char=29,
    end_char=38,
    confidence=1.0
)
entities.append(entity1)

entity2 = create_entity(
    annotation_id=annotation.id,
    text="fever",
    entity_type=EntityType.SYMPTOM,
    start_char=70,
    end_char=75,
    confidence=1.0
)
entities.append(entity2)

entity3 = create_entity(
    annotation_id=annotation.id,
    text="shortness of breath",
    entity_type=EntityType.SYMPTOM,
    start_char=80,
    end_char=99,
    confidence=1.0
)
entities.append(entity3)

# Disease
entity4 = create_entity(
    annotation_id=annotation.id,
    text="pneumonia",
    entity_type=EntityType.DISEASE,
    start_char=155,
    end_char=164,
    confidence=0.95
)
entities.append(entity4)

# Medications
entity5 = create_entity(
    annotation_id=annotation.id,
    text="azithromycin",
    entity_type=EntityType.MEDICATION,
    start_char=178,
    end_char=190,
    confidence=1.0
)
entities.append(entity5)

entity6 = create_entity(
    annotation_id=annotation.id,
    text="500mg",
    entity_type=EntityType.DOSAGE,
    start_char=191,
    end_char=196,
    confidence=1.0
)
entities.append(entity6)

entity7 = create_entity(
    annotation_id=annotation.id,
    text="albuterol",
    entity_type=EntityType.MEDICATION,
    start_char=218,
    end_char=227,
    confidence=1.0
)
entities.append(entity7)

print(f"\nCreated {len(entities)} entities")


2025-11-06 07:21:36,790 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-11-06 07:21:36,792 INFO sqlalchemy.engine.Engine INSERT INTO entities (text, entity_type, start_char, end_char, confidence, annotation_id, created_at, updated_at) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
2025-11-06 07:21:36,793 INFO sqlalchemy.engine.Engine [generated in 0.00085s] ('dry cough', 'SYMPTOM', 29, 38, 1.0, 1, '2025-11-05 21:21:36.790581', '2025-11-05 21:21:36.790594')
2025-11-06 07:21:36,794 INFO sqlalchemy.engine.Engine COMMIT
2025-11-06 07:21:36,797 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-11-06 07:21:36,799 INFO sqlalchemy.engine.Engine SELECT entities.id, entities.text, entities.entity_type, entities.start_char, entities.end_char, entities.confidence, entities.annotation_id, entities.created_at, entities.updated_at 
FROM entities 
WHERE entities.id = ?
2025-11-06 07:21:36,800 INFO sqlalchemy.engine.Engine [generated in 0.00077s] (1,)
✅ Entity created: ID=1, Text='dry cough', Type=symptom
2025-

In [12]:
# Create relations between entities
# Entity IDs: 1=dry cough, 2=fever, 3=shortness of breath, 4=pneumonia, 5=azithromycin, 6=500mg, 7=albuterol

print("Creating relations between entities...\n")

# Symptoms indicate pneumonia
rel1 = create_relation(
    annotation_id=annotation.id,
    source_entity_id=entity1.id,  # dry cough
    target_entity_id=entity4.id,  # pneumonia
    relation_type=RelationType.INDICATES,
    confidence=0.9
)

rel2 = create_relation(
    annotation_id=annotation.id,
    source_entity_id=entity2.id,  # fever
    target_entity_id=entity4.id,  # pneumonia
    relation_type=RelationType.INDICATES,
    confidence=0.95
)

rel3 = create_relation(
    annotation_id=annotation.id,
    source_entity_id=entity3.id,  # shortness of breath
    target_entity_id=entity4.id,  # pneumonia
    relation_type=RelationType.INDICATES,
    confidence=0.95
)

# Medications treat pneumonia
rel4 = create_relation(
    annotation_id=annotation.id,
    source_entity_id=entity5.id,  # azithromycin
    target_entity_id=entity4.id,  # pneumonia
    relation_type=RelationType.TREATS,
    confidence=1.0
)

rel5 = create_relation(
    annotation_id=annotation.id,
    source_entity_id=entity7.id,  # albuterol
    target_entity_id=entity4.id,  # pneumonia
    relation_type=RelationType.TREATS,
    confidence=0.85
)

# Dosage for medication
rel6 = create_relation(
    annotation_id=annotation.id,
    source_entity_id=entity6.id,  # 500mg
    target_entity_id=entity5.id,  # azithromycin
    relation_type=RelationType.DOSAGE_FOR,
    confidence=1.0
)

print("\n✅ All relations created successfully!")


Creating relations between entities...

2025-11-06 07:21:36,877 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-11-06 07:21:36,879 INFO sqlalchemy.engine.Engine INSERT INTO relations (relation_type, source_entity_id, target_entity_id, confidence, annotation_id, created_at, updated_at) VALUES (?, ?, ?, ?, ?, ?, ?)
2025-11-06 07:21:36,880 INFO sqlalchemy.engine.Engine [generated in 0.00123s] ('INDICATES', 1, 4, 0.9, 1, '2025-11-05 21:21:36.877223', '2025-11-05 21:21:36.877255')
2025-11-06 07:21:36,882 INFO sqlalchemy.engine.Engine COMMIT
2025-11-06 07:21:36,885 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-11-06 07:21:36,887 INFO sqlalchemy.engine.Engine SELECT relations.id, relations.relation_type, relations.source_entity_id, relations.target_entity_id, relations.confidence, relations.annotation_id, relations.created_at, relations.updated_at 
FROM relations 
WHERE relations.id = ?
2025-11-06 07:21:36,888 INFO sqlalchemy.engine.Engine [generated in 0.00085s] (1,)
✅ Relation cre

In [13]:
# View the entities we created
print("\n📋 Entities in this annotation:")
df_entities = list_entities(annotation_id=annotation.id)
display(df_entities)



📋 Entities in this annotation:
2025-11-06 07:21:36,945 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-11-06 07:21:36,947 INFO sqlalchemy.engine.Engine SELECT entities.id, entities.text, entities.entity_type, entities.start_char, entities.end_char, entities.confidence, entities.annotation_id, entities.created_at, entities.updated_at 
FROM entities 
WHERE entities.annotation_id = ?
2025-11-06 07:21:36,948 INFO sqlalchemy.engine.Engine [generated in 0.00092s] (1,)
2025-11-06 07:21:36,953 INFO sqlalchemy.engine.Engine ROLLBACK


,ID,Text,Type,Start,End,Confidence,Annotation ID
0,1,dry cough,symptom,29,38,1.00,1
1,2,fever,symptom,70,75,1.00,1
2,3,shortness of breath,symptom,80,99,1.00,1
3,4,pneumonia,disease,155,164,0.95,1
4,5,azithromycin,medication,178,190,1.00,1
5,6,500mg,dosage,191,196,1.00,1
6,7,albuterol,medication,218,227,1.00,1


In [14]:
# View the relations we created
print("\n🔗 Relations in this annotation:")
df_relations = list_relations(annotation_id=annotation.id)
display(df_relations)



🔗 Relations in this annotation:
2025-11-06 07:21:36,978 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-11-06 07:21:36,979 INFO sqlalchemy.engine.Engine SELECT relations.id, relations.relation_type, relations.source_entity_id, relations.target_entity_id, relations.confidence, relations.annotation_id, relations.created_at, relations.updated_at 
FROM relations 
WHERE relations.annotation_id = ?
2025-11-06 07:21:36,979 INFO sqlalchemy.engine.Engine [generated in 0.00057s] (1,)
2025-11-06 07:21:36,982 INFO sqlalchemy.engine.Engine ROLLBACK


,ID,Source Entity ID,Relation Type,Target Entity ID,Confidence,Annotation ID
0,1,1,indicates,4,0.90,1
1,2,2,indicates,4,0.95,1
2,3,3,indicates,4,0.95,1
3,4,5,treats,4,1.00,1
4,5,7,treats,4,0.85,1
5,6,6,dosage_for,5,1.00,1


## Example 2: Create Another Document

Let's create another sample document with its own annotations.


In [15]:
# Create second document
sample_text_2 = """Patient with type 2 diabetes mellitus diagnosed 5 years ago.
Current medications include metformin 1000mg twice daily and glipizide 10mg once daily.
Recent HbA1c results show 7.8%, indicating suboptimal glycemic control.
Patient reports occasional episodes of dizziness and fatigue."""

doc2 = create_document(
    title="Case 2: Diabetes Management",
    text=sample_text_2
)

print(f"\nDocument ID: {doc2.id}")


2025-11-06 07:21:36,992 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-11-06 07:21:36,993 INFO sqlalchemy.engine.Engine INSERT INTO documents (title, text, created_at, updated_at) VALUES (?, ?, ?, ?)
2025-11-06 07:21:36,994 INFO sqlalchemy.engine.Engine [cached since 0.2453s ago] ('Case 2: Diabetes Management', 'Patient with type 2 diabetes mellitus diagnosed 5 years ago. \nCurrent medications include metformin 1000mg twice daily and glipizide 10mg once daily. \nRecent HbA1c results show 7.8%, indicating suboptimal glycemic control. \nPatient reports occasional episodes of dizziness and fatigue.', '2025-11-05 21:21:36.992400', '2025-11-05 21:21:36.992412')
2025-11-06 07:21:36,996 INFO sqlalchemy.engine.Engine COMMIT
2025-11-06 07:21:36,999 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-11-06 07:21:37,000 INFO sqlalchemy.engine.Engine SELECT documents.id, documents.title, documents.text, documents.created_at, documents.updated_at 
FROM documents 
WHERE documents.id = ?
2025-11

In [16]:
# Create annotation for second document
annotation2 = create_annotation(
    document_id=doc2.id,
    annotator_id="doctor_johnson",
    status="in_progress"
)

print(f"\nAnnotation ID: {annotation2.id}")


2025-11-06 07:21:37,009 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-11-06 07:21:37,010 INFO sqlalchemy.engine.Engine INSERT INTO annotations (annotator_id, status, document_id, created_at, updated_at) VALUES (?, ?, ?, ?, ?)
2025-11-06 07:21:37,011 INFO sqlalchemy.engine.Engine [cached since 0.2441s ago] ('doctor_johnson', 'IN_PROGRESS', 2, '2025-11-05 21:21:37.008927', '2025-11-05 21:21:37.008943')
2025-11-06 07:21:37,012 INFO sqlalchemy.engine.Engine COMMIT
2025-11-06 07:21:37,015 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-11-06 07:21:37,016 INFO sqlalchemy.engine.Engine SELECT annotations.id, annotations.annotator_id, annotations.status, annotations.document_id, annotations.created_at, annotations.updated_at 
FROM annotations 
WHERE annotations.id = ?
2025-11-06 07:21:37,017 INFO sqlalchemy.engine.Engine [cached since 0.2431s ago] (2,)
✅ Annotation created: ID=2, Document=2, Status=in_progress
2025-11-06 07:21:37,018 INFO sqlalchemy.engine.Engine ROLLBACK

Annotation

In [17]:
# Add entities to second annotation
text2 = sample_text_2

# Disease
e1 = create_entity(
    annotation_id=annotation2.id,
    text="type 2 diabetes mellitus",
    entity_type=EntityType.DISEASE,
    start_char=16,
    end_char=39,
    confidence=1.0
)

# Medications
e2 = create_entity(
    annotation_id=annotation2.id,
    text="metformin",
    entity_type=EntityType.MEDICATION,
    start_char=104,
    end_char=113,
    confidence=1.0
)

e3 = create_entity(
    annotation_id=annotation2.id,
    text="1000mg",
    entity_type=EntityType.DOSAGE,
    start_char=114,
    end_char=120,
    confidence=1.0
)

e4 = create_entity(
    annotation_id=annotation2.id,
    text="glipizide",
    entity_type=EntityType.MEDICATION,
    start_char=142,
    end_char=151,
    confidence=1.0
)

e5 = create_entity(
    annotation_id=annotation2.id,
    text="10mg",
    entity_type=EntityType.DOSAGE,
    start_char=152,
    end_char=156,
    confidence=1.0
)

# Lab value
e6 = create_entity(
    annotation_id=annotation2.id,
    text="HbA1c",
    entity_type=EntityType.LAB_VALUE,
    start_char=185,
    end_char=190,
    confidence=1.0
)

# Symptoms
e7 = create_entity(
    annotation_id=annotation2.id,
    text="dizziness",
    entity_type=EntityType.SYMPTOM,
    start_char=230,
    end_char=239,
    confidence=0.95
)

e8 = create_entity(
    annotation_id=annotation2.id,
    text="fatigue",
    entity_type=EntityType.SYMPTOM,
    start_char=244,
    end_char=251,
    confidence=0.95
)

print("\n✅ Entities created for second annotation")


2025-11-06 07:21:37,027 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-11-06 07:21:37,028 INFO sqlalchemy.engine.Engine INSERT INTO entities (text, entity_type, start_char, end_char, confidence, annotation_id, created_at, updated_at) VALUES (?, ?, ?, ?, ?, ?, ?, ?)
2025-11-06 07:21:37,029 INFO sqlalchemy.engine.Engine [cached since 0.2368s ago] ('type 2 diabetes mellitus', 'DISEASE', 16, 39, 1.0, 2, '2025-11-05 21:21:37.027168', '2025-11-05 21:21:37.027179')
2025-11-06 07:21:37,030 INFO sqlalchemy.engine.Engine COMMIT
2025-11-06 07:21:37,033 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-11-06 07:21:37,034 INFO sqlalchemy.engine.Engine SELECT entities.id, entities.text, entities.entity_type, entities.start_char, entities.end_char, entities.confidence, entities.annotation_id, entities.created_at, entities.updated_at 
FROM entities 
WHERE entities.id = ?
2025-11-06 07:21:37,035 INFO sqlalchemy.engine.Engine [cached since 0.2364s ago] (8,)
✅ Entity created: ID=8, Text='type 2 di

In [18]:
# Add relations for second annotation
# Medications treat disease
create_relation(
    annotation_id=annotation2.id,
    source_entity_id=e2.id,  # metformin
    target_entity_id=e1.id,  # type 2 diabetes
    relation_type=RelationType.TREATS,
    confidence=1.0
)

create_relation(
    annotation_id=annotation2.id,
    source_entity_id=e4.id,  # glipizide
    target_entity_id=e1.id,  # type 2 diabetes
    relation_type=RelationType.TREATS,
    confidence=1.0
)

# Dosages for medications
create_relation(
    annotation_id=annotation2.id,
    source_entity_id=e3.id,  # 1000mg
    target_entity_id=e2.id,  # metformin
    relation_type=RelationType.DOSAGE_FOR,
    confidence=1.0
)

create_relation(
    annotation_id=annotation2.id,
    source_entity_id=e5.id,  # 10mg
    target_entity_id=e4.id,  # glipizide
    relation_type=RelationType.DOSAGE_FOR,
    confidence=1.0
)

# Symptoms caused by disease
create_relation(
    annotation_id=annotation2.id,
    source_entity_id=e1.id,  # type 2 diabetes
    target_entity_id=e7.id,  # dizziness
    relation_type=RelationType.HAS_SYMPTOM,
    confidence=0.8
)

create_relation(
    annotation_id=annotation2.id,
    source_entity_id=e1.id,  # type 2 diabetes
    target_entity_id=e8.id,  # fatigue
    relation_type=RelationType.HAS_SYMPTOM,
    confidence=0.85
)

print("\n✅ Relations created for second annotation")


2025-11-06 07:21:37,116 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-11-06 07:21:37,117 INFO sqlalchemy.engine.Engine INSERT INTO relations (relation_type, source_entity_id, target_entity_id, confidence, annotation_id, created_at, updated_at) VALUES (?, ?, ?, ?, ?, ?, ?)
2025-11-06 07:21:37,117 INFO sqlalchemy.engine.Engine [cached since 0.2386s ago] ('TREATS', 9, 8, 1.0, 2, '2025-11-05 21:21:37.115841', '2025-11-05 21:21:37.115853')
2025-11-06 07:21:37,119 INFO sqlalchemy.engine.Engine COMMIT
2025-11-06 07:21:37,122 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-11-06 07:21:37,123 INFO sqlalchemy.engine.Engine SELECT relations.id, relations.relation_type, relations.source_entity_id, relations.target_entity_id, relations.confidence, relations.annotation_id, relations.created_at, relations.updated_at 
FROM relations 
WHERE relations.id = ?
2025-11-06 07:21:37,124 INFO sqlalchemy.engine.Engine [cached since 0.2367s ago] (7,)
✅ Relation created: ID=7, 9 --[treats]--> 8
2025-11

## View Complete Database


In [19]:
# Print overall summary
database_summary()


2025-11-06 07:21:37,181 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-11-06 07:21:37,181 INFO sqlalchemy.engine.Engine SELECT documents.id, documents.title, documents.text, documents.created_at, documents.updated_at 
FROM documents
2025-11-06 07:21:37,182 INFO sqlalchemy.engine.Engine [cached since 0.4505s ago] ()
2025-11-06 07:21:37,183 INFO sqlalchemy.engine.Engine SELECT annotations.id, annotations.annotator_id, annotations.status, annotations.document_id, annotations.created_at, annotations.updated_at 
FROM annotations
2025-11-06 07:21:37,183 INFO sqlalchemy.engine.Engine [cached since 0.4495s ago] ()
2025-11-06 07:21:37,184 INFO sqlalchemy.engine.Engine SELECT entities.id, entities.text, entities.entity_type, entities.start_char, entities.end_char, entities.confidence, entities.annotation_id, entities.created_at, entities.updated_at 
FROM entities
2025-11-06 07:21:37,185 INFO sqlalchemy.engine.Engine [cached since 0.4482s ago] ()
2025-11-06 07:21:37,186 INFO sqlalchemy.engin

In [20]:
# View all documents
print("\n📄 ALL DOCUMENTS:")
df_docs = list_documents()
display(df_docs)



📄 ALL DOCUMENTS:
2025-11-06 07:21:37,193 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-11-06 07:21:37,194 INFO sqlalchemy.engine.Engine SELECT documents.id, documents.title, documents.text, documents.created_at, documents.updated_at 
FROM documents
2025-11-06 07:21:37,195 INFO sqlalchemy.engine.Engine [cached since 0.4633s ago] ()
2025-11-06 07:21:37,196 INFO sqlalchemy.engine.Engine ROLLBACK


,ID,Title,Text Preview,Text Length,Created
0,1,Case 1: Respiratory Infection,Patient presents with persistent dry cough for...,306,2025-11-05T21:21:36.746874
1,2,Case 2: Diabetes Management,Patient with type 2 diabetes mellitus diagnose...,285,2025-11-05T21:21:36.992400


In [21]:
# View all annotations
print("\n📝 ALL ANNOTATIONS:")
df_anns = list_annotations()
display(df_anns)



📝 ALL ANNOTATIONS:
2025-11-06 07:21:37,204 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-11-06 07:21:37,205 INFO sqlalchemy.engine.Engine SELECT annotations.id, annotations.annotator_id, annotations.status, annotations.document_id, annotations.created_at, annotations.updated_at 
FROM annotations
2025-11-06 07:21:37,206 INFO sqlalchemy.engine.Engine [cached since 0.4719s ago] ()
2025-11-06 07:21:37,208 INFO sqlalchemy.engine.Engine SELECT entities.id AS entities_id, entities.text AS entities_text, entities.entity_type AS entities_entity_type, entities.start_char AS entities_start_char, entities.end_char AS entities_end_char, entities.confidence AS entities_confidence, entities.annotation_id AS entities_annotation_id, entities.created_at AS entities_created_at, entities.updated_at AS entities_updated_at 
FROM entities 
WHERE ? = entities.annotation_id
2025-11-06 07:21:37,209 INFO sqlalchemy.engine.Engine [generated in 0.00070s] (1,)
2025-11-06 07:21:37,211 INFO sqlalchemy.engine.E

,ID,Document ID,Annotator,Status,Entities,Relations,Updated
0,1,1,doctor_smith,in_progress,7,6,2025-11-05T21:21:36.765203
1,2,2,doctor_johnson,in_progress,8,6,2025-11-05T21:21:37.008943


In [22]:
# View all entities
print("\n🏷️  ALL ENTITIES:")
df_ents = list_entities()
display(df_ents)



🏷️  ALL ENTITIES:
2025-11-06 07:21:37,226 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-11-06 07:21:37,227 INFO sqlalchemy.engine.Engine SELECT entities.id, entities.text, entities.entity_type, entities.start_char, entities.end_char, entities.confidence, entities.annotation_id, entities.created_at, entities.updated_at 
FROM entities
2025-11-06 07:21:37,227 INFO sqlalchemy.engine.Engine [cached since 0.4904s ago] ()
2025-11-06 07:21:37,229 INFO sqlalchemy.engine.Engine ROLLBACK


,ID,Text,Type,Start,End,Confidence,Annotation ID
0,1,dry cough,symptom,29,38,1.00,1
1,2,fever,symptom,70,75,1.00,1
2,3,shortness of breath,symptom,80,99,1.00,1
3,4,pneumonia,disease,155,164,0.95,1
4,5,azithromycin,medication,178,190,1.00,1
5,6,500mg,dosage,191,196,1.00,1
6,7,albuterol,medication,218,227,1.00,1
7,8,type 2 diabetes mellitus,disease,16,39,1.00,2
8,9,metformin,medication,104,113,1.00,2
9,10,1000mg,dosage,114,120,1.00,2


In [23]:
# View all relations
print("\n🔗 ALL RELATIONS:")
df_rels = list_relations()
display(df_rels)



🔗 ALL RELATIONS:
2025-11-06 07:21:37,241 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2025-11-06 07:21:37,241 INFO sqlalchemy.engine.Engine SELECT relations.id, relations.relation_type, relations.source_entity_id, relations.target_entity_id, relations.confidence, relations.annotation_id, relations.created_at, relations.updated_at 
FROM relations
2025-11-06 07:21:37,242 INFO sqlalchemy.engine.Engine [cached since 0.5023s ago] ()
2025-11-06 07:21:37,244 INFO sqlalchemy.engine.Engine ROLLBACK


,ID,Source Entity ID,Relation Type,Target Entity ID,Confidence,Annotation ID
0,1,1,indicates,4,0.90,1
1,2,2,indicates,4,0.95,1
2,3,3,indicates,4,0.95,1
3,4,5,treats,4,1.00,1
4,5,7,treats,4,0.85,1
5,6,6,dosage_for,5,1.00,1
6,7,9,treats,8,1.00,2
7,8,11,treats,8,1.00,2
8,9,10,dosage_for,9,1.00,2
9,10,12,dosage_for,11,1.00,2


## Manual Population Template

Use this template to add more documents and annotations manually.


In [24]:
# TEMPLATE: Create a new document
# Uncomment and modify to create your own document

# my_text = """Your medical text here."""
#
# my_doc = create_document(
#     title="My Case Title",
#     text=my_text
# )
#
# my_annotation = create_annotation(
#     document_id=my_doc.id,
#     annotator_id="your_name",
#     status="in_progress"
# )
#
# # Add entities
# my_entity = create_entity(
#     annotation_id=my_annotation.id,
#     text="entity text",
#     entity_type=EntityType.DISEASE,  # Change to appropriate type
#     start_char=0,  # Find in text
#     end_char=5,    # Find in text
#     confidence=1.0
# )
#
# # Add relations
# my_relation = create_relation(
#     annotation_id=my_annotation.id,
#     source_entity_id=entity1.id,
#     target_entity_id=entity2.id,
#     relation_type=RelationType.TREATS,
#     confidence=1.0
# )


In [25]:
print("\n📋 ENTITY TYPES:")
print("="*50)
for entity_type in EntityType:
    print(f"  • EntityType.{entity_type.name:15} = '{entity_type.value}'")

print("\n🔗 RELATION TYPES:")
print("="*50)
for relation_type in RelationType:
    print(f"  • RelationType.{relation_type.name:20} = '{relation_type.value}'")



📋 ENTITY TYPES:
  • EntityType.DISEASE         = 'disease'
  • EntityType.MEDICATION      = 'medication'
  • EntityType.SYMPTOM         = 'symptom'
  • EntityType.PROCEDURE       = 'procedure'
  • EntityType.ANATOMY         = 'anatomy'
  • EntityType.LAB_VALUE       = 'lab_value'
  • EntityType.DOSAGE          = 'dosage'
  • EntityType.OTHER           = 'other'

🔗 RELATION TYPES:
  • RelationType.TREATS               = 'treats'
  • RelationType.CAUSES               = 'causes'
  • RelationType.HAS_SYMPTOM          = 'has_symptom'
  • RelationType.INDICATES            = 'indicates'
  • RelationType.CONTRAINDICATES      = 'contraindicates'
  • RelationType.DOSAGE_FOR           = 'dosage_for'
  • RelationType.LOCATED_IN           = 'located_in'
  • RelationType.TEMPORAL             = 'temporal'
  • RelationType.OTHER                = 'other'
